# Replay of the 08:00 forecast

Reviewer point A8 asks for a repeatable replay of past days using only what was known at 08:00 on
D-1. It covers input availability, publication delay, revisions and complete target hours on normal
and clock-change days. Reviewer point B3/A5 also asks for model-versus-baseline results by season.

The logic lives in `winterload.forecast.evaluation.backtest` and `winterload.forecast.evaluation.season_report`, which have
unit tests. This notebook reads the feature store and appends results to the result store, so every run
adds to the log. Rows from before `run_at` 2026-10-31 used the 00 UTC weather run, which missed 08:00
in summer, and a smaller store.

It never reads the test year: `history` stops at `VALIDATION_END`.

In [1]:
import winterload.config  # Loads .env before anything imports fsspec.

import dataclasses
import datetime as dt

import pandas as pd
from winterload.forecast.evaluation.backtest import check_day, replay_day, rows_of_local_day, training_cutoff
from winterload.forecast.evaluation.season_report import chronological_forecast, report_by_season
from winterload.forecast.evaluation.splits import TRAIN_END, TRAIN_START, VALIDATION_END
from winterload.forecast.features.dataset import TARGET_COLUMN
from winterload.forecast.lifecycle.retrain_trigger import RetrainTriggerPolicy
from winterload.forecast.metrics import mape
from winterload.forecast.model.baseline import seasonal_naive_forecast
from winterload.forecast.model.preliminary import PreliminaryModel
from winterload.repository.factory import result_repository
from winterload.repository.results import PUBLICATION_LOG, REPLAY_LOG, REVISION_LOG, SEASON_REPORT, WEATHER_RUN_LOG
from winterload.repository.training_frame import load_dataframe
from winterload.repository.weather import WeatherRepository
from winterload.storage import get_storage

RUN_AT = pd.Timestamp.now(tz="UTC").floor("s")

SUMMER_WEEK = [dt.date(2025, 7, 1) + dt.timedelta(days=i) for i in range(7)]
WINTER_WEEK = [dt.date(2025, 1, 13) + dt.timedelta(days=i) for i in range(7)]
SPRING_CLOCK_CHANGE = dt.date(2025, 3, 30)
AUTUMN_CLOCK_CHANGE = dt.date(2024, 10, 27)
REPLAY_DAYS = [AUTUMN_CLOCK_CHANGE, SPRING_CLOCK_CHANGE, *SUMMER_WEEK, *WINTER_WEEK]

storage = get_storage()
weather = WeatherRepository(storage)
results = result_repository()


def append_result(name, rows):
    results.append(name, rows.assign(run_at=RUN_AT))
    return rows


## History outside the test year

Hours without weather drop out, so these counts show how much of each split the model can use. The Single Runs
archive starts on 2024-03-14, which sets the first usable day.

In [2]:
history = load_dataframe(storage, TRAIN_START, VALIDATION_END)
train_rows = int((history["event_timestamp"] < pd.Timestamp(TRAIN_END, tz="UTC")).sum())
print(f"{len(history)} rows with load and weather: {train_rows} in TRAIN, {len(history) - train_rows} in VALIDATION")

13201 rows with load and weather: 11329 in TRAIN, 1872 in VALIDATION


## Replaying each day

Each replay trains only on labels from before local midnight on D-1, and `check_day` confirms that the stored weather
run was published before 08:00 on D-1.

Columns:
- `target_day` is the local day forecast, and `issue_time` is 08:00 Zurich time on D-1, when the forecast goes out.
- `expected_hours` is the number of hours in the local day (23, 24 or 25), and `present_hours` the hours with load and
  weather in the store. `complete` is true when both match.
- `weather_run` is the start (UTC) of the weather run behind the day's rows. `weather_run_available` is true when that
  start plus the assumed 8h publication delay falls before `issue_time`.
- `publication_tolerance_hours` is how late the newest load hour that the 48h lag needs may arrive and still make it
  into the forecast.
- `training_rows` counts the hourly rows the replay model trained on, all before local midnight on D-1.
- `model_mape`, `model_mae`, `baseline_mape` and `baseline_mae` score the day's hours, for the model and for the
  seasonal-naive baseline (the load one week earlier). MAPE is a fraction (0.05 = 5%), MAE is in MW.

In [3]:
def replay(day):
    day_frame = rows_of_local_day(history, day)
    check = check_day(day, day_frame, weather.runs_of_day(day))
    row = {
        "target_day": day,
        "issue_time": check.issue_time,
        "expected_hours": check.expected_hours,
        "present_hours": check.present_hours,
        "complete": check.complete,
        "weather_run": check.weather_run,
        "weather_run_available": check.weather_run_available,
        "publication_tolerance_hours": check.publication_tolerance / pd.Timedelta(1, unit="hours"),
    }
    if day_frame.empty:
        return row, None
    train = history.loc[history["event_timestamp"] < training_cutoff(day)]
    result = replay_day(PreliminaryModel().fit(train), day_frame)
    row |= {
        "training_rows": len(train),
        "model_mape": result.model_mape,
        "model_mae": result.model_mae,
        "baseline_mape": result.baseline_mape,
        "baseline_mae": result.baseline_mae,
    }
    return row, day_frame.assign(forecast=result.forecast)


log_rows, frames = [], {}
for day in REPLAY_DAYS:
    row, frames[day] = replay(day)
    log_rows.append(row)
append_result(REPLAY_LOG, pd.DataFrame(log_rows))

,target_day,issue_time,expected_hours,present_hours,complete,weather_run,weather_run_available,publication_tolerance_hours,training_rows,model_mape,model_mae,baseline_mape,baseline_mae
0,2024-10-27,2024-10-26 08:00:00+02:00,25,25,True,2024-10-25 18:00:00+00:00,True,8.0,5375,0.095610,525.727010,0.212213,1163.187200
1,2025-03-30,2025-03-29 08:00:00+01:00,23,23,True,2025-03-28 18:00:00+00:00,True,10.0,9072,0.035193,231.574624,0.037952,251.953478
2,2025-07-01,2025-06-30 08:00:00+02:00,24,24,True,2025-06-29 18:00:00+00:00,True,9.0,11303,0.063612,495.555922,0.067908,464.263750
3,2025-07-02,2025-07-01 08:00:00+02:00,24,24,True,2025-06-30 18:00:00+00:00,True,9.0,11327,0.088803,658.542393,0.043807,322.067917
4,2025-07-03,2025-07-02 08:00:00+02:00,24,24,True,2025-07-01 18:00:00+00:00,True,9.0,11351,0.056554,375.921367,0.043100,294.530000
5,2025-07-04,2025-07-03 08:00:00+02:00,24,24,True,2025-07-02 18:00:00+00:00,True,9.0,11375,0.061334,458.692042,0.045296,323.578333
6,2025-07-05,2025-07-04 08:00:00+02:00,24,24,True,2025-07-03 18:00:00+00:00,True,9.0,11399,0.052941,385.644581,0.051298,300.802083
7,2025-07-06,2025-07-05 08:00:00+02:00,24,24,True,2025-07-04 18:00:00+00:00,True,9.0,11423,0.060491,362.358843,0.063058,370.250833
8,2025-07-07,2025-07-06 08:00:00+02:00,24,24,True,2025-07-05 18:00:00+00:00,True,9.0,11447,0.025576,153.669319,0.069985,466.530833
9,2025-01-13,2025-01-12 08:00:00+01:00,24,24,True,2025-01-11 18:00:00+00:00,True,9.0,7248,0.108956,980.821982,0.102507,916.999583


## Retrain trigger over the replayed weeks


`winterload.pipeline.monitoring.retrain_monitor` runs this policy on live data. This cell shows the decision over the
two replayed weeks, one in each season.

Each line prints the decision for one week: `model_mape` and `baseline_mape` over the week's hours, `threshold_mape`
= 0.9 x `baseline_mape`, and `retrain`, which is true when the model's MAPE exceeds the threshold.

In [4]:
for name, days in [("summer week", SUMMER_WEEK), ("winter week", WINTER_WEEK)]:
    week = pd.concat([frames[day] for day in days])
    actual = week[TARGET_COLUMN].to_numpy()
    print(name, RetrainTriggerPolicy().decide(mape(actual, week["forecast"].to_numpy()),
                                              mape(actual, seasonal_naive_forecast(week))))

summer week RetrainDecision(retrain=True, model_mape=0.05847315388979206, baseline_mape=0.05492175020852748, threshold_mape=0.04942957518767473)
winter week RetrainDecision(retrain=False, model_mape=0.05818767897381244, baseline_mape=0.07287301374880371, threshold_mape=0.06558571237392334)


## Publication delay and revisions

`winterload.pipeline.monitoring.publication_probe` logs one observation per source to the result store on every run of
`uv run winterload -p monitoring`. These cells read what it logged. The load table comes from ENTSO-E and the proposal
names Energy-Charts, so the probe checks both.

Publication log: `source` is the load source. `observed_at` is when the probe ran, `latest_published_hour` the newest
hour the source had published by then, and `delay_hours` the gap between the two. `tolerance_hours` is the smallest
publication tolerance of any day (8h, on the autumn clock change), and `within_tolerance` says whether the delay stayed
within it. `run_at` is when the row was written.

Weather run log: `host` is the Open-Meteo host. `run` is the start of the newest run it served, `available_at` when it
published that run, and `delay_hours` the gap between the two. `within_assumed_delay` compares that gap with the 8h the
replay assumes.

In [5]:
results.read(PUBLICATION_LOG).tail(6)

,source,observed_at,latest_published_hour,delay_hours,tolerance_hours,within_tolerance,run_at
10,entsoe,2026-10-31 12:58:46+00:00,2026-10-31 11:00:00+00:00,1.979444,8.0,True,2026-10-31 12:58:46+00:00
11,energy_charts,2026-10-31 12:58:46+00:00,2026-10-31 09:00:00+00:00,3.979444,8.0,True,2026-10-31 12:58:46+00:00
12,entsoe,2026-10-31 13:33:13+00:00,2026-10-31 12:00:00+00:00,1.553611,8.0,True,2026-10-31 13:33:13+00:00
13,energy_charts,2026-10-31 13:33:13+00:00,2026-10-31 09:00:00+00:00,4.553611,8.0,True,2026-10-31 13:33:13+00:00
14,entsoe,2026-11-01 20:30:00+00:00,2026-11-01 18:00:00+00:00,2.500000,8.0,True,2026-11-01 20:30:00+00:00
15,energy_charts,2026-11-01 20:30:00+00:00,2026-11-01 16:00:00+00:00,4.500000,8.0,True,2026-11-01 20:30:00+00:00


In [6]:
results.read(WEATHER_RUN_LOG).tail(6)

,host,observed_at,run,available_at,delay_hours,within_assumed_delay,run_at
8,https://single-runs-api.open-meteo.com,2026-10-31 12:58:46+00:00,2026-10-31 06:00:00+00:00,2026-10-31 12:05:27+00:00,6.090833,True,2026-10-31 12:58:46+00:00
9,https://api.open-meteo.com,2026-10-31 12:58:46+00:00,2026-10-31 00:00:00+00:00,2026-10-31 06:51:49+00:00,6.863611,True,2026-10-31 12:58:46+00:00
10,https://single-runs-api.open-meteo.com,2026-10-31 13:33:13+00:00,2026-10-31 06:00:00+00:00,2026-10-31 12:05:27+00:00,6.090833,True,2026-10-31 13:33:13+00:00
11,https://api.open-meteo.com,2026-10-31 13:33:13+00:00,2026-10-31 06:00:00+00:00,2026-10-31 13:26:02+00:00,7.433889,False,2026-10-31 13:33:13+00:00
12,https://single-runs-api.open-meteo.com,2026-11-01 20:30:00+00:00,2026-11-01 12:00:00+00:00,2026-11-01 18:25:21+00:00,6.422500,True,2026-11-01 20:30:00+00:00
13,https://api.open-meteo.com,2026-11-01 20:30:00+00:00,2026-11-01 12:00:00+00:00,2026-11-01 18:56:30+00:00,6.941667,True,2026-11-01 20:30:00+00:00


The probe also compares each source with the value it first saw, and a stored ENTSO-E week with a fresh download.

Columns: `window` names the comparison. `earlier snapshot` compares each hour with the value the probe first saw for
it, and `stored ... to ...` compares the stored ENTSO-E week with a fresh download. `compared_hours` counts the hours
present in both versions, `changed_hours` the hours whose value differs, and `max_abs_change_mw` the largest
difference in MW.

In [7]:
results.read(REVISION_LOG).tail(6)

,source,window,compared_hours,changed_hours,max_abs_change_mw,run_at
11,energy_charts,earlier snapshot,69,0,0.0,2026-10-31 12:58:46+00:00
12,entsoe,earlier snapshot,71,0,0.0,2026-10-31 13:33:13+00:00
13,energy_charts,earlier snapshot,69,0,0.0,2026-10-31 13:33:13+00:00
14,entsoe,earlier snapshot,41,0,0.0,2026-11-01 20:30:00+00:00
15,energy_charts,earlier snapshot,38,0,0.0,2026-11-01 20:30:00+00:00
16,entsoe,stored 2025-07-01 to 2025-07-07,168,0,0.0,2026-11-01 20:30:00+00:00


## Results by season

Each month is scored by a model trained on the rows before it. The `months` column shows what each season rests on.

Columns: `hours` counts the scored hourly rows of a season, and `months` lists the local months they come from. Every
hour of those months is scored, not only the replayed days, so the counts run into the thousands: a full winter
(November to February) holds 120 days, or 2,880 hours. Scoring starts in July 2024, because the weather archive starts
on 2024-03-14 and each month needs 90 days of earlier rows to train on. In this run, August 2025 lacks the six days
from 2025-08-06 to 2025-08-11, which have no complete weather, and September 2025 ends with the validation period on
22 September. `model_*` and `baseline_*` score those hours for the model and the seasonal-naive baseline. MAPE is a fraction (0.05 = 5%), MAE is in MW.

In [8]:
report = pd.DataFrame([dataclasses.asdict(result) for result in report_by_season(chronological_forecast(history))])
append_result(SEASON_REPORT, report.rename(columns={"segment": "season"}).assign(months=report["months"].str.join(" ")))

,season,hours,months,model_mape,model_mae,baseline_mape,baseline_mae
0,winter,2880,2024-11 2024-12 2025-01 2025-02,0.087811,614.226943,0.101208,714.401264
1,transition,4202,2024-09 2024-10 2025-03 2025-04 2025-05 2025-09,0.056613,363.109962,0.075847,490.857702
2,summer,3552,2024-07 2024-08 2025-06 2025-07 2025-08,0.076057,463.500251,0.079625,486.268682
